In [1]:
import numpy as np
import cv2

In [2]:
face_cascade=cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

In [3]:
#read image with alpha channel
#cv2.IMREAD_UNCHANGED flag is used to read the image with alpha channel
sunglasses=cv2.imread('ss.png',cv2.IMREAD_UNCHANGED)

In [4]:
Cap=cv2.VideoCapture(0)
if sunglasses is None:
    raise FileNotFoundError("ss.png not loaded. Check the file path and rerun cell 2.")
while True:
    flag,frame =Cap.read()
    if not flag:
        print("Error: Could not read frame")
        break
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray,1.3)
    for x,y,w,h in faces:
        overlay_width=w
        overlay_height=int(h*0.5) #height =~ 50 percent of face
        #resize the sunglasses image to fit on face
        if sunglasses is not None:
            resized_sunglasses = cv2.resize(sunglasses, (overlay_width, overlay_height))
        else:
            continue
        #positioning the sunglasses
        y_offset=y+h//4 #roughly eye region 
        x_offset=x
        
        #handle transparency (Alpha Blending)
        #split pbg into color channel
        overlay_img = resized_sunglasses[:,:,:3] #BGR channels
         #last channel-alpha transparency
        mask = resized_sunglasses[:,:,3] #Alpha channel as mask
        #create inverse mask
        mask_inv = cv2.bitwise_not(mask)
        #region of interest(ROI) on the frame where we want to place the sunglasses
        roi=frame[y_offset:y_offset+overlay_height,x_offset:x_offset+overlay_width]
        bg =cv2.bitwise_and(roi,roi,mask=mask_inv)
        #extract foreground
        fg =cv2.bitwise_and(overlay_img,overlay_img,mask=mask)
        #combine background and foreground
        combined = cv2.add(bg,fg)
        #place the combined image back to the original frame
        frame[y_offset:y_offset+overlay_height,x_offset:x_offset+overlay_width]=combined
        cv2.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),2)
        
    cv2.imshow('Sunglasses Filter',frame)
    key = cv2.waitKey(1)& 0xFF
    if key ==27:
        break
Cap.release()
cv2.destroyAllWindows()



        
       





KeyboardInterrupt: 